# Liquid-S4

Hasani, Lechner, Wang, Chahine, Amini, Rus, *Liquid Structural State-Space Models*, ICLR 2023 ([arXiv:2209.12951](https://arxiv.org/abs/2209.12951)).

A diagonal S4-style linear recurrence `x_k = A x_{k-1} + B u_k` whose transition `A` is gated by the input -- see `model.py` (`LiquidS4Layer`) and `../../papers/README.md`.

This notebook trains a `LiquidS4Model` for ETTh1 long-horizon forecasting and plots a sample forecast.

In [ ]:
import sys
sys.path.insert(0, '../..')
sys.path.insert(0, '.')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from liquid_playground.data import load_ett
from liquid_playground.device import resolve_device
from liquid_playground.utils.seed import set_seed
from model import LiquidS4Model

set_seed(0)
device = resolve_device('auto')  # or 'cpu' / 'cuda' / 'mps'
print('device:', device)

In [ ]:
seq_len, pred_len = 96, 24
train_x, train_y, test_x, test_y = load_ett(seq_len=seq_len, pred_len=pred_len)
n_channels = train_x.shape[-1]
train_x, test_x = train_x.to(device), test_x.to(device)
train_y_flat = train_y.reshape(train_y.shape[0], -1).to(device)
test_y_flat = test_y.reshape(test_y.shape[0], -1).to(device)
print(train_x.shape, train_y_flat.shape)

In [ ]:
model = LiquidS4Model(input_size=n_channels, state_size=32, output_size=pred_len * n_channels).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

batch_size = 64
n_train = train_x.shape[0]
history = {'train_mse': [], 'test_mse': []}
epochs = 15
for epoch in range(epochs):
    model.train()
    perm = torch.randperm(n_train, device=device)
    total = 0.0
    for i in range(0, n_train, batch_size):
        idx = perm[i:i + batch_size]
        opt.zero_grad()
        pred = model(train_x[idx])
        loss = loss_fn(pred, train_y_flat[idx])
        loss.backward()
        opt.step()
        total += loss.item() * len(idx)
    model.eval()
    with torch.no_grad():
        test_mse = loss_fn(model(test_x), test_y_flat).item()
    history['train_mse'].append(total / n_train)
    history['test_mse'].append(test_mse)

print(f"final test MSE: {history['test_mse'][-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history['train_mse'], label='train'); axes[0].plot(history['test_mse'], label='test')
axes[0].set_title('MSE'); axes[0].set_xlabel('epoch'); axes[0].legend()

model.eval()
with torch.no_grad():
    sample_pred = model(test_x[:1]).reshape(pred_len, n_channels).cpu()
sample_true = test_y[0].cpu()
axes[1].plot(sample_true[:, 0], label='true (channel 0)')
axes[1].plot(sample_pred[:, 0], label='predicted (channel 0)')
axes[1].set_title('sample forecast'); axes[1].legend()
fig.tight_layout()
plt.show()